# Avance 4. Modelos alternativos

Este notebook consolida los modelos alternativos que sí consideramos, resume qué se probó, qué no funcionó, qué sí mejoró y cierra con una decisión explícita de modelo individual para la siguiente etapa.

**Decisión de este avance:** el modelo seleccionado para continuar es **TSViT + Projector + Qwen2.5-3B + LoRA**.

**Nota de alcance**
- Este notebook está pensado para lectura en GitHub.
- No todos los notebooks fuente conservan métricas finales completas; en esos casos se referencian como antecedentes y no se usan para el ranking principal.
- Los archivos .pt no estan incluidos en el repositorio por su tamaño, si se necesitan los podemos cargar a google drive


## 1. Objetivo y criterio de comparación

El objetivo de este avance fue comparar varias rutas de modelado para clasificación de cultivos con PASTIS, con tres preguntas concretas:

1. ¿Qué familias de modelos dan mejor desempeño medible en el problema actual?
2. ¿Qué ajustes finos sí movieron la métrica principal?
3. ¿Cuál es el mejor candidato para continuar, considerando no solo la métrica, sino también la fidelidad a la tarea y la capacidad temporal/semántica?

### Criterios usados en este notebook
- Separamos los resultados por **régimen experimental** para no comparar tareas incompatibles como si fueran el mismo problema.
- En PASTIS patch-level usamos `mAP` o `mAP_combined` como referencia principal cuando está disponible.
- En PASTIS24 + LLM usamos `accuracy` y `macro F1`, porque el problema allí es clasificación monoclase.
- El tiempo de entrenamiento solo se reporta cuando quedó medido en outputs guardados; en el resto se muestra `N/D`.


## 2. Notebooks y artefactos fuente

### Notebooks referenciados
- [Avance3_BeMyEyes_v2.ipynb](Avance3_BeMyEyes_v2.ipynb)
- [ModeloPruebaDualViT.ipynb](ModeloPruebaDualViT.ipynb)
- [prueba_CLIP_v1.ipynb](prueba_CLIP_v1.ipynb)
- [prueba_CLIP_v2.ipynb](prueba_CLIP_v2.ipynb)
- [Avance5_TSViT_LLM_PASTIS.ipynb](../Avance5_TSViT_LLM_PASTIS.ipynb)
- [farSLIP.ipynb](farSLIP.ipynb)
- [prueba_hibrido_v01.ipynb](prueba_hibrido_v01.ipynb)

### Modelos fuente que sí se referencian, pero no entran al ranking principal

| Notebook | Estado en este reporte | Motivo |
|---|---:|---|
| `Avance3_BeMyEyes_v2` | Referencia cualitativa | No conserva una tabla final de métricas comparable con el resto |
| `ModeloPruebaDualViT` | Referencia metodológica | No conserva métricas finales en outputs visibles |
| `prueba_CLIP_v2` | Métricas parciales | Sí conserva retrieval, `Macro F1` y tiempo medido, pero no un `mAP` final explícito |

## 3. Comparativa principal

### 3.1 Patch completo / PASTIS multilabel

En este bloque la tarea es la más cercana al problema original: patch completo, varios cultivos por imagen y evaluación multilabel.

| Modelo | Dataset / régimen | Métrica principal | Métrica 2 | Métrica 3 |
|---|---|---:|---:|---:|
| **FarSLIP v01** | PASTIS patch completo | `mAP = 0.468` | `Macro F1 = 0.451` | `Recall@1 = 0.046` |
| **FarSLIP v02 ajustado** | PASTIS patch completo | `mAP_combined = 0.449` | `Macro F1 = 0.501` | `Recall@1 = 0.003`  |

### 3.2 Monocrop / parcela única

Aquí simplificamos el problema a chips con una sola parcela/clase objetivo. Este bloque no reemplaza al benchmark principal, pero sí sirve para entender si la mezcla de cultivos estaba degradando la alineación.

| Modelo | Dataset / régimen | Métrica principal | Métrica 2 | Métrica 3 |
|---|---|---:|---:|---:|
| **FarSLIP 10 bandas monocrop + global fix** | PASTIS monocrop top-5 clases | `mAP_combined = 0.598` | `Macro F1 = 0.485` | `Top-1 acc = 0.623` |
| **FarSLIP 10 bandas monocrop** | PASTIS monocrop top-5 clases | `mAP_combined = 0.503` | `Macro F1 = 0.322` | `Top-1 acc = 0.470` | 
| **CLIP RGB monocrop** | PASTIS monocrop top-5 clases | `mAP_combined = 0.200` | `Macro F1 = 0.111` | `Top-1 acc = 0.384` |
| **CLIP 10 bandas monocrop** | PASTIS monocrop top-5 clases | `mAP_combined = 0.200` | `Macro F1 = 0.111` | `Top-1 acc = 0.384` |

### 3.3 PASTIS24 + LLM

Este bloque trabaja sobre `PASTIS24` preprocesado como clasificación monoclase con una arquitectura temporal + proyector + LLM. No es directamente comparable con el patch-level anterior, pero sí es el mejor candidato para la siguiente fase del proyecto.

| Modelo | Dataset / régimen | Métrica principal | Métrica 2 | Métrica 3 |
|---|---|---:|---:|---:|
| **TSViT + Projector + Qwen2.5-3B + LoRA** | PASTIS24 clasificación | `Test acc = 0.768` | `Test Macro F1 = 0.622` | `Best val acc = 0.790` |
| **TSViT standalone** | PASTIS24 clasificación | `Best val acc = 0.621` | `Best val F1 = 0.533` | `Epochs = 50` |


## 4. Métricas parciales preservadas de antecedentes

Estas corridas sí aportan contexto, pero no se usan como base del ranking principal porque el notebook no conserva una evaluación final homogénea con el resto.

| Modelo / notebook | Señal cuantitativa preservada | Lectura útil |
|---|---|---|
| **CLIP v2** (`prueba_CLIP_v2`) | `Recall@1 = 0.054`, `Recall@5 = 0.210`, `Recall@10 = 0.359`, `Macro F1 = 0.500`, `Wall time = 27h 2m 32s` | Sirve como baseline fuerte de retrieval y clasificación por prompts |
| **BeMyEyes v2** | Evidencia arquitectónica y pipeline multimodal | Útil como antecedente VLM, no como base del modelo final |
| **DualViT** | Implementación exploratoria sin métricas finales visibles | Se conserva como línea metodológica, no como candidato final |
| **CLIP v1** | Checkpoint preservado, evaluación final no consolidada | Se usa solo como antecedente del salto a `CLIP v2` |


## 5. Qué probamos y no funcionó

### 5.1 Cambios de caption en patch-level
- Se probaron variantes como `satellite view of agricultural fields with {crop}`.
- El efecto fue marginal o nulo.
- Ejemplo directo: `v1_baseline` quedó en `mAP_combined = 0.4177` y `v3_satellite_fields` en `0.4175`.
- Conclusión: el cuello de botella no estaba solo en el wording del texto.

### 5.2 CLIP monocrop
- Se esperaba que reducir la ambigüedad de cultivos mezclados ayudara a la rama global.
- No ocurrió: `CLIP RGB` y `CLIP 10ch` quedaron colapsados alrededor de `mAP_combined ≈ 0.20`.
- Conclusión: en este dataset, solo limpiar el recorte no basta para que CLIP aprenda una señal global útil.

### 5.3 Reanudar más epochs en FarSLIP v02
- Extender `stage2` más allá del mejor checkpoint no mejoró de forma consistente.
- Las corridas reanudadas no superaron el mejor valor ya encontrado.
- Conclusión: el mayor retorno vino del tuning correcto de la receta, no de seguir sumando epochs.


## 6. Qué sí funcionó

### 6.1 Ajuste fino de FarSLIP v02
- El baseline largo de la línea v02 estaba alrededor de `mAP_combined = 0.374`.
- Después de tuning y mejor calibración de prompts/evaluación, el mejor checkpoint útil quedó en `mAP_combined = 0.449` con `Macro F1 = 0.501`.
- Conclusión: sí hubo mejora real y medible en la línea FarSLIP patch-level.

### 6.2 Experimento monocrop como diagnóstico espacial
- `FarSLIP 10ch monocrop` subió hasta `0.503`.
- `FarSLIP 10ch monocrop + global fix` llegó a `0.598`.
- Conclusión: cuando aislamos una sola parcela, la rama espacial/local sí aprende mejor. Esto valida la hipótesis de que la mezcla de cultivos en un patch completo daña parte de la señal.

### 6.3 TSViT + Projector + Qwen
- `TSViT standalone` alcanzó `Best val acc = 0.621`.
- Al conectar `TSViT -> Projector -> Qwen2.5-3B + LoRA`, la validación subió a `0.790` y el test quedó en `accuracy = 0.768`, `Macro F1 = 0.622`.
- Conclusión: esta línea sí muestra una ganancia fuerte y coherente al añadir un decoder semántico más potente encima de un encoder temporal adecuado.


## 7. Evidencia visual

### 7.1 Híbrido temporal previo

![Curvas híbrido v01](assets/hibrido_v01_loss_curves.png)

![Similitud híbrido v01](assets/hibrido_v01_similarity.png)

### 7.2 FarSLIP v01

![Curvas FarSLIP v01](assets/farslip_v01_loss_curves.png)

![Per-crop FarSLIP v01](assets/farslip_v01_per_crop.png)

### 7.3 TSViT + Projector + Qwen

![Resumen test TSViT + Qwen](assets/tsvit_qwen_test_report.png)

![Ejemplos TSViT + Qwen](assets/tsvit_qwen_examples.png)


## 8. Modelo seleccionado

### Modelo individual elegido

**`TSViT + Projector + Qwen2.5-3B + LoRA`**

### Justificación
- Es el modelo que mejor combina **señal temporal**, **adaptación semántica** y **capacidad de generalización** hacia la siguiente etapa.
- Supera claramente a `TSViT standalone`: pasa de `Best val acc = 0.621` a `Best val acc = 0.790`.
- En test mantiene una señal fuerte: `accuracy = 0.768`, `Macro F1 = 0.622`.
- Arquitectónicamente es una evolución más rica que `CLIP` o `FarSLIP` global, porque no solo compara embeddings, sino que proyecta tokens visuales temporales hacia un decoder lingüístico con LoRA.
- Para el objetivo del proyecto, este modelo es más prometedor que quedarse únicamente con monocrop o con patch-level contrastivo.

### Qué no significa esta decisión
- No significa que el mejor `mAP` monocrop deje de ser útil; monocrop fue importante como diagnóstico.
- No significa que `FarSLIP v02` haya sido un fracaso; sí mejoró respecto a su baseline.
- Sí significa que, para la **siguiente etapa**, la línea con mayor potencial es la temporal + projector + LLM.


## 9. Siguientes pasos

1. Consolidar `TSViT + Projector + Qwen` como línea principal y repetir en un protocolo más estable de train/val/test.
2. Reincorporar aprendizajes de monocrop para mejorar la limpieza de supervisión espacial.
3. Conservar `FarSLIP v02` como baseline patch-level fuerte para comparar cualquier nueva arquitectura.
4. Mantener `CLIP v2` como baseline histórico de retrieval y texto-imagen.
